In [1]:
!pip install transformers==4.46.0 accelerate==1.1.1 -U bitsandbytes

In [5]:
import torch
import gc

from transformers import AutoModelForCausalLM, BitsAndBytesConfig


MODEL = "Qwen/Qwen2.5-1.5B-Instruct"


def resident_vram_gb():
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved() / (1024 ** 3)


def load(dtype):
    if dtype == "fp16":
        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            torch_dtype=torch.float16,
            device_map="cuda",
        )
    elif dtype == "int8":
        quant_config = BitsAndBytesConfig(load_in_8bit=True)
        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            quantization_config=quant_config,
            device_map="cuda",
        )
    elif dtype == "int4":
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            quantization_config=quant_config,
            device_map="cuda",
        )
    else:
        raise ValueError(f"Unknown dtype: {dtype}")


results = {}

for dtype in ["fp16", "int8", "int4"]:
    model = load(dtype)

    vram = resident_vram_gb()
    results[dtype] = vram
    print(dtype, round(vram, 2), "GB")

    # Delete in the same scope that created the reference
    del model # BUG
    gc.collect()
    torch.cuda.empty_cache()


# ---- Verify ----
fp16_gb = results["fp16"]
int8_gb = results["int8"]
int4_gb = results["int4"]

assert int8_gb < fp16_gb
assert int4_gb < int8_gb

print("GREEN CHECK: PASS")

fp16 3.06 GB
int8 1.74 GB
int4 1.15 GB
GREEN CHECK: PASS


In [6]:
fp16_gb = results["fp16"]
int8_gb = results["int8"]
int4_gb = results["int4"]

assert int8_gb < fp16_gb
assert int4_gb < int8_gb

print("GREEN CHECK: PASS")

GREEN CHECK: PASS


In [7]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="cuda",
)

tok_ids = torch.randint(0, 1000, (1, 64)).to("cuda")

leaked_outputs = []
leak_samples = []

for i in range(20):
    # Intentionally no torch.no_grad()
    out = model(tok_ids)

    # Intentionally keep every output
    leaked_outputs.append(out.logits)

    row = {
        "iter": i,
        "reserved_mb": round(reserved_mb(), 1),
    }

    leak_samples.append(row)

    if i % 5 == 0:
        print(row)

{'iter': 0, 'reserved_mb': 3254.0}
{'iter': 5, 'reserved_mb': 4294.0}
{'iter': 10, 'reserved_mb': 5354.0}
{'iter': 15, 'reserved_mb': 6414.0}


In [8]:
import numpy as np


def detect_leak(samples_mb, slope_threshold_mb_per_iter=1.0):
    x = np.arange(len(samples_mb))
    y = np.array(samples_mb)

    slope, intercept = np.polyfit(x, y, 1)

    leaking = slope > slope_threshold_mb_per_iter

    return {
        "slope_mb_per_iter": round(float(slope), 3),
        "threshold_mb_per_iter": slope_threshold_mb_per_iter,
        "leaking": bool(leaking),
        "n_samples": len(samples_mb),
    }


leak_result = detect_leak(
    [s["reserved_mb"] for s in leak_samples]
)

print(leak_result)
assert leak_result["leaking"]

{'slope_mb_per_iter': 211.248, 'threshold_mb_per_iter': 1.0, 'leaking': True, 'n_samples': 20}


In [9]:
del model
del leaked_outputs
del out

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="cuda",
)

fixed_samples = []

with torch.no_grad():
    for i in range(20):
        out = model(tok_ids)

        # Use a scalar value. Do not save the GPU tensor.
        _ = out.logits.sum().item()

        row = {
            "iter": i,
            "reserved_mb": round(reserved_mb(), 1),
        }

        fixed_samples.append(row)

fixed_result = detect_leak(
    [s["reserved_mb"] for s in fixed_samples]
)

print(fixed_result)
assert not fixed_result["leaking"]

{'slope_mb_per_iter': 0.286, 'threshold_mb_per_iter': 1.0, 'leaking': False, 'n_samples': 20}


In [10]:
import json


report = {
    "reload_loop_baseline": samples,
    "leaky_run": leak_result,
    "fixed_run": fixed_result,
    "leaky_samples": [s["reserved_mb"] for s in leak_samples],
    "fixed_samples": [s["reserved_mb"] for s in fixed_samples],
}

with open("leak_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

{
  "reload_loop_baseline": [
    {
      "cycle": 0,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 1,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 2,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 3,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 4,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    }
  ],
  "leaky_run": {
    "slope_mb_per_iter": 211.248,
    "threshold_mb_per_iter": 1.0,
    "leaking": true,
    "n_samples": 20
  },
  "fixed_run": {
    "slope_mb_per_iter": 0.286,
    "threshold_mb_per_iter": 1.0,
    "leaking": false,
    "n_samples": 20
  },
  "leaky_samples": [
    3254.0,
    3462.0,
    3670.0,
    3878.0,
    4086.0,
    4294.0,
    4522.0,
    4730.0,
    4938.0,
    5146.0,
    5354.0,
    5562.0,
    5790.0,
    5998.0,
    6206.0,
    6414.0,
    6622.0,
    6830.0,
    7058.0,
 

In [11]:
import json
import numpy as np
import sys


REPORT_PATH = "leak_report.json"
SLOPE_THRESHOLD_MB_PER_ITER = 1.0


def refit_slope(samples_mb):
    x = np.arange(len(samples_mb))
    y = np.array(samples_mb)
    slope, intercept = np.polyfit(x, y, 1)
    return float(slope)


def main():
    try:
        with open(REPORT_PATH, "r") as f:
            report = json.load(f)
    except FileNotFoundError:
        print(f"GREEN CHECK: FAIL (missing {REPORT_PATH})")
        sys.exit(1)

    required_keys = [
        "reload_loop_baseline",
        "leaky_run",
        "fixed_run",
        "leaky_samples",
        "fixed_samples",
    ]
    for key in required_keys:
        if key not in report:
            print(f"GREEN CHECK: FAIL (missing key: {key})")
            sys.exit(1)

    leaky_samples = report["leaky_samples"]
    fixed_samples = report["fixed_samples"]

    if len(leaky_samples) < 5 or len(fixed_samples) < 5:
        print("GREEN CHECK: FAIL (not enough samples)")
        sys.exit(1)

    leaky_slope = refit_slope(leaky_samples)
    fixed_slope = refit_slope(fixed_samples)

    leaky_ok = leaky_slope > SLOPE_THRESHOLD_MB_PER_ITER
    fixed_ok = fixed_slope <= SLOPE_THRESHOLD_MB_PER_ITER

    print(f"Refit leaky slope: {round(leaky_slope, 3)} MB/iter")
    print(f"Refit fixed slope: {round(fixed_slope, 3)} MB/iter")

    if leaky_ok and fixed_ok:
        print("GREEN CHECK: PASS")
    else:
        if not leaky_ok:
            print("Reason: leaky_samples did not show a real leak")
        if not fixed_ok:
            print("Reason: fixed_samples still show a leak")
        print("GREEN CHECK: FAIL")
        sys.exit(1)


if __name__ == "__main__":
    main()

Refit leaky slope: 211.248 MB/iter
Refit fixed slope: 0.286 MB/iter
GREEN CHECK: PASS
